In [ ]:
# libraries
import os
import json
import time
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import nbformat
from IPython.display import display

In [ ]:
# globals; run through run_pipeline.ipynb
ROOT = Path(os.environ['NCA_PROJECT_ROOT'])
TASK = ROOT / 'e_phase1_test'
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CFG = json.loads((INPUT / 'config.json').read_text())
DEVICE = torch.device(CFG['device'])
torch.set_num_threads(CFG['cpu_threads'])
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

In [ ]:
# functions passed through c_models/output; definitions only, no training side effects
runtime_nb = nbformat.read(INPUT / 'runtime.ipynb', as_version=4)
for runtime_cell in runtime_nb.cells:
    exec(compile(runtime_cell.source, 'runtime.ipynb', 'exec'), globals())
PACK = torch.load(INPUT / 'dataset.pt', weights_only=True, map_location='cpu')
SPECS = CFG['models']

In [ ]:
# run and condition imports
manifest = pd.read_csv(INPUT / 'phase1_manifest.csv')
conditions = pd.read_csv(INPUT / 'conditions.csv')
metric_frames, dynamics_frames, channel_rows, equivariance_rows, examples = [], [], [], [], []

In [ ]:
# held-out reconstruction and external transmission errors
with torch.no_grad():
    for run in manifest.to_dict('records'):
        spec = next(item for item in SPECS if item['name'] == run['model'])
        model, checkpoint = load_trained(INPUT / 'phase1_runs' / run['checkpoint'], spec, CFG, DEVICE, run['replicate'])
        generator = torch_stream(CFG, DEVICE, 50 + run['replicate'])
        metadata = {key: run[key] for key in ['model', 'regime', 'replicate']}
        steps = steps_for(model, run['regime'], CFG['eval_steps'])
        for split in ['test_composition', 'test_angle', 'test_color', 'test_translation', 'test_joint']:
            ids = panel_ids(PACK, split, CFG)
            p, u = np.zeros(len(ids)), np.full(len(ids), 0.5)
            target, _ = select_target(PACK, ids, p, u, DEVICE)
            for rate in CFG['stress_flip_rates']:
                for trial in range(CFG['channel_trials']):
                    started = time.perf_counter()
                    state = generate_panel(model, PACK, CFG, ids, p, u, run['regime'], generator, flips=all_flips(rate))
                    if DEVICE.type == 'cuda':
                        torch.cuda.synchronize()
                    row_metadata = {**metadata, 'split': split, 'condition': ids, 'flip_rate': rate, 'trial': trial,
                                    'intervention': 'none', 'steps': steps, 'bits': bit_cost(model, steps),
                                    'seconds_per_image': (time.perf_counter() - started) / len(ids)}
                    metric_frames.append(metric_rows(state, target, row_metadata))
                    if rate == 0 and trial == 0:
                        examples.append({**metadata, 'split': split, 'condition': int(ids[0]),
                                         'target': target[0].cpu(), 'generated': state[0, ..., :4].cpu()})
        display(pd.DataFrame([{**metadata, 'test': 'reconstruction and error sweep', 'status': 'complete'}]))

In [ ]:
# channel interventions, persistence, repair and missing-seed control
with torch.no_grad():
    for run in manifest.to_dict('records'):
        spec = next(item for item in SPECS if item['name'] == run['model'])
        model, _ = load_trained(INPUT / 'phase1_runs' / run['checkpoint'], spec, CFG, DEVICE, run['replicate'])
        generator = torch_stream(CFG, DEVICE, 60 + run['replicate'])
        metadata = {key: run[key] for key in ['model', 'regime', 'replicate']}
        ids = panel_ids(PACK, 'test_composition', CFG)[:CFG['batch_size']]
        p, u = np.zeros(len(ids)), np.full(len(ids), 0.5)
        target, _ = select_target(PACK, ids, p, u, DEVICE)
        cue = make_cue(PACK, ids, p, u, CFG, DEVICE)
        steps = steps_for(model, run['regime'], CFG['eval_steps'])
        state, frames = run_steps(model, seed_state(PACK, ids, CFG, DEVICE), cue, 2 * steps, generator,
                                  record=sorted(set([max(1, steps // 2), steps, steps + steps // 2, 2 * steps])))
        for step, frame in frames.items():
            dynamics_frames.append(metric_rows(frame.to(DEVICE), target, {**metadata, 'condition': ids,
                                   'intervention': 'persistence', 'steps': step, 'bits': bit_cost(model, step)}))
        grown = frames[steps].to(DEVICE)
        channel_rows.extend(channel_diagnostics(model, grown, cue, generator, metadata))
        damaged = centered_damage(grown, PACK['coordinates'][ids], CFG)
        dynamics_frames.append(metric_rows(damaged, target, {**metadata, 'condition': ids, 'intervention': 'damage_initial',
                               'steps': 0, 'bits': 0}))
        repaired, _ = run_steps(model, damaged, cue, steps, generator)
        dynamics_frames.append(metric_rows(repaired, target, {**metadata, 'condition': ids, 'intervention': 'repair',
                               'steps': steps, 'bits': bit_cost(model, steps)}))
        channels = [('m',)] if spec['family'] == 'message' else [('q',), ('r',), ('q', 'r')]
        for muted in channels:
            muted_state = generate_panel(model, PACK, CFG, ids, p, u, run['regime'], generator, mute=muted)
            metric_frames.append(metric_rows(muted_state, target, {**metadata, 'condition': ids, 'split': 'test_composition',
                                 'flip_rate': 0.0, 'trial': 0, 'intervention': 'mute_' + ''.join(muted), 'steps': steps,
                                 'bits': bit_cost(model, steps)}))
        blank_state = generate_panel(model, PACK, CFG, ids, p, u, run['regime'], generator, blank=True)
        metric_frames.append(metric_rows(blank_state, target, {**metadata, 'condition': ids, 'split': 'test_composition',
                             'flip_rate': 0.0, 'trial': 0, 'intervention': 'remove_seed_and_cue', 'steps': steps,
                             'bits': bit_cost(model, steps)}))

In [ ]:
# transformation consistency; score both consistency and target fit so blank outputs cannot look successful
base_id = int(conditions.query('angle == 0 and dx == 0 and dy == 0 and saturation == 1 and red_gain == 1')['condition'].iloc[0])
with torch.no_grad():
    for run in manifest.to_dict('records'):
        spec = next(item for item in SPECS if item['name'] == run['model'])
        model, _ = load_trained(INPUT / 'phase1_runs' / run['checkpoint'], spec, CFG, DEVICE, run['replicate'])
        generator = torch_stream(CFG, DEVICE, 70 + run['replicate'])
        metadata = {key: run[key] for key in ['model', 'regime', 'replicate']}
        canonical = generate_panel(model, PACK, CFG, [base_id], [0.0], [0.5], run['regime'], generator)[..., :4].clamp(0, 1)
        for condition in panel_ids(PACK, 'test_angle', CFG)[:3]:
            row = conditions.iloc[int(condition)]
            state = generate_panel(model, PACK, CFG, [condition], [0.0], [0.5], run['regime'], generator)
            expected = transform_rgba(canonical, row['angle'], row['dx'], row['dy'], row['saturation'],
                                       [row['red_gain'], row['green_gain'], row['blue_gain']])
            target = PACK['targets'][condition, 0].float().to(DEVICE)[None]
            equivariance_rows.append({**metadata, 'condition': int(condition), 'angle': row['angle'],
                                      'consistency_mse': float((state[..., :4].clamp(0, 1) - expected).square().mean()),
                                      'target_mse': float(image_metrics(state, target)['mse'][0])})

In [ ]:
# exports
pd.concat(metric_frames, ignore_index=True).to_csv(OUTPUT / 'phase1_metrics.csv', index=False)
pd.concat(dynamics_frames, ignore_index=True).to_csv(OUTPUT / 'phase1_dynamics.csv', index=False)
pd.DataFrame(channel_rows).to_csv(OUTPUT / 'phase1_channels.csv', index=False)
pd.DataFrame(equivariance_rows).to_csv(OUTPUT / 'phase1_transform_consistency.csv', index=False)
torch.save(examples, OUTPUT / 'phase1_examples.pt')
display(pd.concat(metric_frames).query("intervention == 'none' and flip_rate == 0").groupby(['model', 'regime', 'split'])[
        ['mse', 'foreground_rmse', 'alpha_iou']].mean().reset_index())